# SymptoGraph-LLM: download the datasets straight into Google Drive

Run this in Colab. It downloads all eight open datasets on Colab's own network
(fast), then writes them to your Drive. Nothing is downloaded or uploaded from
your own machine.

**How it works.** The data is first staged on the Colab VM's local disk, which is
much faster than writing through the Drive mount. The Reddit Mental Health
Dataset is then slimmed from 3.14 GB to 0.41 GB by dropping 346
precomputed tf-idf and readability columns that this project does not use, and
only the result is copied to Drive.

| | Downloaded to the VM | Ends up in Drive |
|---|---|---|
| Default (`KEEP_RAW_RMHD = False`) | ~3.1 GB | **~0.6 GB** |
| Keeping the raw CSVs (`True`) | ~3.1 GB | ~3.7 GB |

Expect roughly 10–20 minutes, mostly the Drive copy. A free Drive account has
15 GB, so the default comfortably fits.

If the runtime disconnects, just run the cells again. Downloads resume: finished
files are skipped and every Reddit Mental Health file is checked against its
Zenodo checksum.

## 1. Mount Drive and set the paths

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Where the data should end up (inside your Drive)
DRIVE_DIR = '/content/drive/MyDrive/SymptoGraph-LLM/data'
# Fast local scratch space on the Colab VM
STAGE_DIR = '/content/stage'
# Set to True to also keep the full 3 GB raw Reddit CSVs in Drive
KEEP_RAW_RMHD = False

os.environ.update(DRIVE_DIR=DRIVE_DIR, STAGE_DIR=STAGE_DIR,
                  DATA_DIR=STAGE_DIR, JOBS='8')
os.makedirs(STAGE_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
print('staging in', STAGE_DIR, '-> Drive folder', DRIVE_DIR)

## 2. Get the download scripts

The repo itself is tiny, so it is cloned to the VM rather than to Drive.

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q $REPO $CHECKOUT

# A clone left over from an earlier session can be missing these, which silently
# sends the data to the wrong folder. Check before downloading 3 GB.
need = ['scripts/download_data.sh', 'scripts/slim_rmhd.py']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
supports_data_dir = 'DATA_DIR' in open(f'{CHECKOUT}/scripts/download_data.sh').read()
assert not missing, f'stale checkout, missing {missing} - delete {CHECKOUT} and re-run'
assert supports_data_dir, f'stale download_data.sh - delete {CHECKOUT} and re-run'

!git -C $CHECKOUT log --oneline -1
print('scripts OK:', need)

## 3. Download every dataset to the VM

Eight datasets: DepressionEmo, Reddit Mental Health, DepSeverity,
CSSRS-Reddit-500, IMHI, Dreaddit, IRF and LoST.

Useful variants, if you want to cut this down:

```python
os.environ['RMHD_FILTER'] = 'depression|anxiety|bipolar|suicidewatch'  # only these subreddits
os.environ['SKIP'] = 'imhi'                                           # skip a dataset entirely
```

In [ ]:
!bash /content/SymptoGraph-LLM/scripts/download_data.sh

got = sorted(os.listdir(STAGE_DIR))
assert 'reddit_mental_health' in got, (
    f'nothing in {STAGE_DIR}; DATA_DIR was ignored. Check for a stale checkout.')
print('\nin', STAGE_DIR, '->', got)

## 4. Slim the Reddit Mental Health Dataset

Keeps only `subreddit`, `author`, `date` and `post`, gzipped. This is the step
that turns 3.14 GB into 0.41 GB (108 files, 1,107,302 posts). Skip this cell if you set
`KEEP_RAW_RMHD = True` and want the raw features.

In [ ]:
!python /content/SymptoGraph-LLM/scripts/slim_rmhd.py \
    --src "$STAGE_DIR/reddit_mental_health" \
    --dst "$STAGE_DIR/reddit_mental_health_slim"

## 5. Copy to Drive

`rsync` can be re-run safely: it only copies what is missing or changed.

In [ ]:
exclude = '' if KEEP_RAW_RMHD else "--exclude=reddit_mental_health/"
!rsync -ah --info=progress2 $exclude "$STAGE_DIR/" "$DRIVE_DIR/"
print('\nIn Drive now:')
!du -sh "$DRIVE_DIR"/* | sort -h
!du -sh "$DRIVE_DIR"

## 6. Check the data loads

Reads one file from each dataset out of Drive and prints its shape, so you know
the copy is intact before you rely on it.

In [ ]:
# Checks every dataset is present and that row counts match the published figures.
# Add --deep to also count all 1,107,302 Reddit MH posts (slower over Drive).
!python /content/SymptoGraph-LLM/scripts/verify_data.py --data "$DRIVE_DIR"

## Using the data in later sessions

Mount Drive and read straight from `DRIVE_DIR`. Reading through the Drive mount
is slow, so for training copy what you need onto the VM's local disk first:

```python
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/data
!rsync -ah /content/drive/MyDrive/SymptoGraph-LLM/data/ /content/data/
```

Reminders on terms of use:

- Keep the data in a **private** Drive folder and do not share it publicly.
  DepressionEmo is "All Rights Reserved" and is licensed for research only.
- Paraphrase any example posts you quote in the paper, and do not attempt to
  identify users.
- `data/README.md` in the repo lists every dataset's source, licence and role.

## Troubleshooting

**The data went to `/content/SymptoGraph-LLM/data` instead of `/content/stage`,
and `slim_rmhd.py` was not found.**

The checkout in that session was older than the scripts. The download itself is
fine, so do not repeat it. Point the remaining steps at wherever the data landed:

```python
import os
os.environ['SRC'] = '/content/SymptoGraph-LLM/data'   # where the data actually is
os.environ['DRIVE_DIR'] = '/content/drive/MyDrive/SymptoGraph-LLM/data'
os.makedirs(os.environ['DRIVE_DIR'], exist_ok=True)

!git -C /content/SymptoGraph-LLM pull -q || true
!test -f /content/SymptoGraph-LLM/scripts/slim_rmhd.py || curl -sSL \
    -o /content/SymptoGraph-LLM/scripts/slim_rmhd.py \
    https://raw.githubusercontent.com/sid-2k6/SymptoGraph-LLM/main/scripts/slim_rmhd.py

!python /content/SymptoGraph-LLM/scripts/slim_rmhd.py \
    --src "$SRC/reddit_mental_health" --dst "$SRC/reddit_mental_health_slim"

!rsync -ah --info=progress2 --exclude=reddit_mental_health/ "$SRC/" "$DRIVE_DIR/"
```

**`rsync: change_dir "/content/stage" failed`** means the staging folder was never
created, so cell 1 did not run. Run the cells in order from the top.

**Starting completely fresh.** Remove the checkout so the clone is current, and
note that anything on the VM (not in Drive) is lost when the runtime recycles:

```python
!rm -rf /content/SymptoGraph-LLM
```
**Checking what is in Drive.** Folder sizes show as `-` in the Drive web view, so
verify with:

```python
!python /content/SymptoGraph-LLM/scripts/verify_data.py \
    --data /content/drive/MyDrive/SymptoGraph-LLM/data
```